# 14 - Scoring (Aurora + AGE)

Compute expertise scores, PageRank, centrality, and betweenness centrality on the
Aurora Serverless v2 + Apache AGE knowledge graph. All graph reads/writes go through
the RDS Data API (`boto3` `rds-data` client) — no direct TCP connections.

**Algorithm overview**

| Metric | Method |
|---|---|
| Expertise score | Sum of `HAS_SKILL` relationship weights per Person |
| In-degree centrality | Count of all incoming edges per Person |
| PageRank | Pure-Python iterative (20 iters, d=0.85) |
| Betweenness centrality | Exact BFS from every node (Python dict graph) |
| Composite score | expertise×0.40 + centrality×0.20 + pagerank×0.30 + betweenness×0.10 |

## Step 1 — Setup

Load `config.json`, build boto3 clients, define the `_cypher()` helper that wraps
every AGE query in the `ag_catalog.cypher(...)` SQL shim.

In [ ]:
import boto3
import json
import time
from pathlib import Path
from collections import defaultdict, deque

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
config = json.loads(Path("config.json").read_text())

REGION       = config["REGION"]
PREFIX       = config.get("PROJECT_PREFIX", "orgiq")
CLUSTER_ARN  = config["AURORA_CLUSTER_ARN"]
SECRET_ARN   = config["AURORA_SECRET_ARN"]
DB_NAME      = config.get("DB_NAME", "orgiq")
GRAPH_NAME   = config.get("GRAPH_NAME", "orgiq_graph")

if not CLUSTER_ARN:
    raise RuntimeError("AURORA_CLUSTER_ARN not set. Run 03-aurora-setup.ipynb first.")

# ---------------------------------------------------------------------------
# Clients
# ---------------------------------------------------------------------------
rds_data = boto3.client("rds-data", region_name=REGION)
dynamo   = boto3.client("dynamodb", region_name=REGION)

# ---------------------------------------------------------------------------
# AGE helper
# ---------------------------------------------------------------------------
def _cypher(query: str) -> list:
    """
    Execute an openCypher query via Apache AGE and return rows as lists of
    plain Python values (strings, numbers, None).  Each cell is extracted
    from the RDS Data API `Field` union dict.
    """
    sql = (
        f"SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', "
        f"$$ {query} $$) as (result ag_catalog.agtype)"
    )
    resp = rds_data.execute_statement(
        resourceArn=CLUSTER_ARN,
        secretArn=SECRET_ARN,
        database=DB_NAME,
        sql=sql,
    )
    rows = []
    for row in resp.get("records", []):
        cells = [list(f.values())[0] for f in row if f]
        rows.append(cells)
    return rows


def _cypher_multi(query: str) -> list:
    """
    Like _cypher() but the AGE projection returns multiple columns.
    Columns are declared in the SQL wrapper; the helper returns raw records.
    Use when you need more than one return value per row.
    """
    resp = rds_data.execute_statement(
        resourceArn=CLUSTER_ARN,
        secretArn=SECRET_ARN,
        database=DB_NAME,
        sql=query,
    )
    rows = []
    for row in resp.get("records", []):
        cells = [list(f.values())[0] for f in row if f]
        rows.append(cells)
    return rows


print(f"\u2713 Config loaded  (prefix={PREFIX}, db={DB_NAME}, graph={GRAPH_NAME})")
print(f"\u2713 Clients ready  (rds-data, dynamodb)")

## Step 2 — Weighted Expertise Score

Sum the `weight` property on every `HAS_SKILL` edge leaving a Person node.
Defaults to 1.0 when the edge has no weight set.  The result is written back
to the Person node as `expertise_score`.

In [ ]:
t0 = time.time()

# ---- 2a: Compute expertise scores ----------------------------------------
expertise_read = """
MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill)
WITH p, sum(coalesce(r.weight, 1.0)) as expertise_score
RETURN p.id, p.name, expertise_score
ORDER BY expertise_score DESC
LIMIT 20
"""

# AGE RETURN with multiple values needs explicit column aliases in SQL
sql_expertise_read = f"""
SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
  MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill)
  WITH p, sum(coalesce(r.weight, 1.0)) as expertise_score
  RETURN p.id as pid, p.name as pname, expertise_score
  ORDER BY expertise_score DESC LIMIT 20
$$) as t(pid ag_catalog.agtype, pname ag_catalog.agtype, expertise_score ag_catalog.agtype)
"""

rows = _cypher_multi(sql_expertise_read)

# Build a dict for write-back: {person_id -> expertise_score}
expertise_scores = {}
print(f"{'Rank':<5} {'Name':<24} {'Expertise Score':<18}")
print("-" * 48)
for i, (pid, pname, score) in enumerate(rows, 1):
    pid_clean   = str(pid).strip('"')   if pid   is not None else ""
    pname_clean = str(pname).strip('"') if pname is not None else "(unknown)"
    score_val   = float(score)           if score is not None else 0.0
    expertise_scores[pid_clean] = score_val
    print(f"{i:<5} {pname_clean:<24} {score_val:<18.3f}")

# ---- 2b: Write expertise_score back to each Person node ------------------
written = 0
for pid, score in expertise_scores.items():
    _cypher(
        f"MATCH (p:Person {{id: '{pid}'}}) SET p.expertise_score = {score:.6f}"
    )
    written += 1

elapsed = time.time() - t0
print(f"\n\u2713 expertise_score written to {written} Person nodes  ({elapsed:.2f}s)")

## Step 3 — In-Degree Centrality

Replaces Neptune Analytics degree centrality.  Count **all incoming edges** per
Person node using an `OPTIONAL MATCH` (so Persons with no incoming edges get 0)
and write the result back as `p.centrality`.

In [ ]:
t0 = time.time()

sql_centrality = f"""
SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
  MATCH (p:Person)
  OPTIONAL MATCH (p)<-[r]-()
  WITH p, count(r) as centrality
  RETURN p.id as pid, p.name as pname, centrality
  ORDER BY centrality DESC LIMIT 20
$$) as t(pid ag_catalog.agtype, pname ag_catalog.agtype, centrality ag_catalog.agtype)
"""

rows = _cypher_multi(sql_centrality)

centrality_scores = {}
print(f"{'Rank':<5} {'Name':<24} {'In-Degree Centrality':<22}")
print("-" * 52)
for i, (pid, pname, cent) in enumerate(rows, 1):
    pid_clean   = str(pid).strip('"')   if pid   is not None else ""
    pname_clean = str(pname).strip('"') if pname is not None else "(unknown)"
    cent_val    = int(cent)              if cent  is not None else 0
    centrality_scores[pid_clean] = cent_val
    print(f"{i:<5} {pname_clean:<24} {cent_val:<22}")

# Write back
for pid, cent in centrality_scores.items():
    _cypher(
        f"MATCH (p:Person {{id: '{pid}'}}) SET p.centrality = {cent}"
    )

elapsed = time.time() - t0
print(f"\n\u2713 centrality written to {len(centrality_scores)} Person nodes  ({elapsed:.2f}s)")

## Step 4 — Python Iterative PageRank

Replaces `neptune.algo.pageRank`.  The full graph (Person + Skill nodes,
`HAS_SKILL` + `WORKED_WITH` edges) is pulled into Python dicts and iterated
for 20 steps with damping factor **d = 0.85**.

$$PR(A) = \frac{1-d}{N} + d \sum_{B \rightarrow A} \frac{PR(B)}{L(B)}$$

where $L(B)$ is the out-degree of node $B$.

In [ ]:
t0 = time.time()

# ---- 4a: Pull all Person node IDs ----------------------------------------
sql_persons = f"""
SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
  MATCH (p:Person)
  RETURN p.id as pid
$$) as t(pid ag_catalog.agtype)
"""
person_rows = _cypher_multi(sql_persons)
all_nodes = [str(r[0]).strip('"') for r in person_rows if r[0] is not None]

# ---- 4b: Pull edges (HAS_SKILL + WORKED_WITH) ----------------------------
# We include Skill node ids so PageRank propagates through them
sql_skills = f"""
SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
  MATCH (s:Skill)
  RETURN s.id as sid
$$) as t(sid ag_catalog.agtype)
"""
skill_rows = _cypher_multi(sql_skills)
all_nodes += [str(r[0]).strip('"') for r in skill_rows if r[0] is not None]
all_nodes = list(dict.fromkeys(all_nodes))  # deduplicate, preserve order

sql_edges_hs = f"""
SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
  MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill)
  RETURN p.id as src, s.id as dst
$$) as t(src ag_catalog.agtype, dst ag_catalog.agtype)
"""
sql_edges_ww = f"""
SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
  MATCH (p:Person)-[r:WORKED_WITH]->(q:Person)
  RETURN p.id as src, q.id as dst
$$) as t(src ag_catalog.agtype, dst ag_catalog.agtype)
"""
edges_hs = _cypher_multi(sql_edges_hs)
edges_ww = _cypher_multi(sql_edges_ww)
edges = [
    (str(s).strip('"'), str(d).strip('"'))
    for s, d in (edges_hs + edges_ww)
    if s is not None and d is not None
]

print(f"Graph: {len(all_nodes)} nodes, {len(edges)} edges")

# ---- 4c: Build adjacency structures -------------------------------------
out_edges  = defaultdict(list)   # node -> [targets]
in_edges   = defaultdict(list)   # node -> [sources]
for src, dst in edges:
    out_edges[src].append(dst)
    in_edges[dst].append(src)

# ---- 4d: Iterative PageRank ---------------------------------------------
N = len(all_nodes)
d = 0.85
ITERATIONS = 20

pr = {node: 1.0 / N for node in all_nodes}

for iteration in range(ITERATIONS):
    new_pr = {}
    for node in all_nodes:
        rank_sum = 0.0
        for src in in_edges[node]:
            out_deg = len(out_edges[src])
            if out_deg > 0:
                rank_sum += pr[src] / out_deg
        new_pr[node] = (1.0 - d) / N + d * rank_sum
    pr = new_pr
    if (iteration + 1) % 5 == 0:
        print(f"  Iteration {iteration+1:>2}/{ITERATIONS} done")

# ---- 4e: Show top Person PageRank values --------------------------------
person_id_set = {str(r[0]).strip('"') for r in person_rows if r[0] is not None}
person_pr = {nid: v for nid, v in pr.items() if nid in person_id_set}
top_pr = sorted(person_pr.items(), key=lambda x: x[1], reverse=True)[:20]

print(f"\n{'Rank':<5} {'Person ID':<30} {'PageRank':<14}")
print("-" * 50)
for i, (nid, score) in enumerate(top_pr, 1):
    print(f"{i:<5} {nid:<30} {score:<14.8f}")

# ---- 4f: Write pagerank back to AGE Person nodes ------------------------
for nid, score in person_pr.items():
    _cypher(
        f"MATCH (p:Person {{id: '{nid}'}}) SET p.pagerank = {score:.10f}"
    )

elapsed = time.time() - t0
print(f"\n\u2713 pagerank written to {len(person_pr)} Person nodes  ({elapsed:.2f}s)")

## Step 5 — Betweenness Centrality Approximation

Replaces `neptune.algo.betweennessCentrality`.  For graphs with fewer than
1 000 nodes, exact betweenness is computed in Python using Brandes' BFS
algorithm.  The graph adjacency is pulled from AGE once and held in a dict.

**Brandes' algorithm** (Brandes 2001): runs a BFS from each source node,
accumulates pair-dependency scores, and back-propagates them along shortest-path
DAG edges.  Time complexity O(VE).

In [ ]:
t0 = time.time()

# ---- 5a: Build undirected adjacency for betweenness (Person nodes only) --
# We treat the graph as undirected for betweenness so isolated people still
# participate via their shared skills.

# Pull Person<->Person edges (WORKED_WITH)
# Plus Person<->Person via shared Skill (co-skill edges, projected)
sql_co_skill = f"""
SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
  MATCH (a:Person)-[:HAS_SKILL]->(s:Skill)<-[:HAS_SKILL]-(b:Person)
  WHERE a.id <> b.id
  RETURN DISTINCT a.id as aid, b.id as bid
$$) as t(aid ag_catalog.agtype, bid ag_catalog.agtype)
"""
co_skill_rows = _cypher_multi(sql_co_skill)

person_ids = list(person_id_set)
pid_index  = {nid: i for i, nid in enumerate(person_ids)}

# undirected adjacency list over person nodes
adj = defaultdict(set)
for row in edges_ww:
    a, b = str(row[0]).strip('"'), str(row[1]).strip('"')
    if a in pid_index and b in pid_index:
        adj[a].add(b)
        adj[b].add(a)
for row in co_skill_rows:
    a, b = str(row[0]).strip('"'), str(row[1]).strip('"')
    if a in pid_index and b in pid_index:
        adj[a].add(b)
        adj[b].add(a)

n_persons = len(person_ids)
print(f"Betweenness graph: {n_persons} Person nodes")
if n_persons > 1000:
    print("\u26a0\ufe0f  Graph too large for exact betweenness — sampling 1000 sources")
    import random
    sources = random.sample(person_ids, 1000)
    scale   = n_persons / 1000.0
else:
    sources = person_ids
    scale   = 1.0

# ---- 5b: Brandes' algorithm ----------------------------------------------
betweenness = defaultdict(float)

for s in sources:
    # BFS
    stack    = []
    pred     = defaultdict(list)    # predecessors on shortest paths
    sigma    = defaultdict(float)   # number of shortest paths from s
    dist     = defaultdict(lambda: -1)
    sigma[s] = 1.0
    dist[s]  = 0
    queue    = deque([s])

    while queue:
        v = queue.popleft()
        stack.append(v)
        for w in adj[v]:
            if dist[w] < 0:            # first visit
                queue.append(w)
                dist[w] = dist[v] + 1
            if dist[w] == dist[v] + 1: # on a shortest path
                sigma[w]  += sigma[v]
                pred[w].append(v)

    # Back-propagation
    delta = defaultdict(float)
    while stack:
        w = stack.pop()
        for v in pred[w]:
            delta[v] += (sigma[v] / sigma[w]) * (1.0 + delta[w])
        if w != s:
            betweenness[w] += delta[w]

# Normalize (undirected: divide by 2)
norm = 1.0 / max(1, (n_persons - 1) * (n_persons - 2) / 2.0)
betweenness_scores = {nid: betweenness[nid] * norm * scale for nid in person_ids}

top_betw = sorted(betweenness_scores.items(), key=lambda x: x[1], reverse=True)[:20]
print(f"\n{'Rank':<5} {'Person ID':<30} {'Betweenness':<14}")
print("-" * 50)
for i, (nid, score) in enumerate(top_betw, 1):
    print(f"{i:<5} {nid:<30} {score:<14.8f}")

# ---- 5c: Write back to AGE -----------------------------------------------
for nid, score in betweenness_scores.items():
    _cypher(
        f"MATCH (p:Person {{id: '{nid}'}}) SET p.betweenness = {score:.10f}"
    )

elapsed = time.time() - t0
print(f"\n\u2713 betweenness written to {len(betweenness_scores)} Person nodes  ({elapsed:.2f}s)")

## Step 6 — Composite Score

Normalise all four metrics to \[0, 1\] (min-max scaling) then combine with weights:

$$\text{composite} = 0.40 \times \hat{E} + 0.20 \times \hat{C} + 0.30 \times \hat{PR} + 0.10 \times \hat{B}$$

where $\hat{x}$ denotes the min-max normalised value of metric $x$.

In [ ]:
t0 = time.time()

# ---- 6a: Fetch all four raw metrics for every Person node ----------------
sql_all_metrics = f"""
SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
  MATCH (p:Person)
  RETURN p.id as pid,
         p.name as pname,
         coalesce(p.expertise_score, 0.0) as exp,
         coalesce(p.centrality,      0.0) as cent,
         coalesce(p.pagerank,        0.0) as pr,
         coalesce(p.betweenness,     0.0) as betw
$$) as t(
  pid  ag_catalog.agtype,
  pname ag_catalog.agtype,
  exp  ag_catalog.agtype,
  cent ag_catalog.agtype,
  pr   ag_catalog.agtype,
  betw ag_catalog.agtype
)
"""
metric_rows = _cypher_multi(sql_all_metrics)

records = []
for row in metric_rows:
    pid, pname, exp, cent, pr_val, betw = row
    records.append({
        "id":         str(pid).strip('"')   if pid   is not None else "",
        "name":       str(pname).strip('"') if pname is not None else "(unknown)",
        "expertise":  float(exp)             if exp   is not None else 0.0,
        "centrality": float(cent)            if cent  is not None else 0.0,
        "pagerank":   float(pr_val)          if pr_val is not None else 0.0,
        "betweenness":float(betw)            if betw  is not None else 0.0,
    })

def _minmax(values):
    lo, hi = min(values), max(values)
    span = hi - lo or 1.0
    return [(v - lo) / span for v in values]

if records:
    exp_norm  = _minmax([r["expertise"]   for r in records])
    cent_norm = _minmax([r["centrality"]  for r in records])
    pr_norm   = _minmax([r["pagerank"]    for r in records])
    betw_norm = _minmax([r["betweenness"] for r in records])

    for i, rec in enumerate(records):
        rec["composite"] = (
            0.40 * exp_norm[i]
            + 0.20 * cent_norm[i]
            + 0.30 * pr_norm[i]
            + 0.10 * betw_norm[i]
        )

# ---- 6b: Write composite_score back to AGE -------------------------------
for rec in records:
    _cypher(
        f"MATCH (p:Person {{id: '{rec['id']}'}}) SET p.composite_score = {rec['composite']:.10f}"
    )

# ---- 6c: Print leaderboard -----------------------------------------------
ranked = sorted(records, key=lambda r: r["composite"], reverse=True)
print(f"{'Rank':<5} {'Name':<22} {'Composite':<11} {'Expertise':<11} {'Centrality':<12} {'PageRank':<11} {'Betweenness':<12}")
print("-" * 85)
for i, rec in enumerate(ranked, 1):
    print(
        f"{i:<5} {rec['name']:<22} {rec['composite']:<11.4f} "
        f"{rec['expertise']:<11.3f} {rec['centrality']:<12.1f} "
        f"{rec['pagerank']:<11.8f} {rec['betweenness']:<12.8f}"
    )

elapsed = time.time() - t0
print(f"\n\u2713 composite_score written to {len(records)} Person nodes  ({elapsed:.2f}s)")

## Step 7 — Per-Skill Leaderboards

For each Skill node in the graph, show the top-5 Persons by `composite_score`
who hold that skill.

In [ ]:
# Fetch all skills
sql_skills_list = f"""
SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
  MATCH (s:Skill)
  RETURN s.id as sid, s.name as sname
  ORDER BY sname
$$) as t(sid ag_catalog.agtype, sname ag_catalog.agtype)
"""
skill_list = _cypher_multi(sql_skills_list)

print(f"Per-Skill Leaderboards ({len(skill_list)} skills)\n")

for sid_raw, sname_raw in skill_list:
    sid   = str(sid_raw).strip('"')   if sid_raw   is not None else ""
    sname = str(sname_raw).strip('"') if sname_raw is not None else sid

    sql_lb = f"""
    SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
      MATCH (p:Person)-[:HAS_SKILL]->(s:Skill {{id: '{sid}'}})
      RETURN p.name as pname, coalesce(p.composite_score, 0.0) as cscore
      ORDER BY cscore DESC
      LIMIT 5
    $$) as t(pname ag_catalog.agtype, cscore ag_catalog.agtype)
    """
    lb_rows = _cypher_multi(sql_lb)

    if lb_rows:
        print(f"  {sname}:")
        for rank, (pname_raw, cscore_raw) in enumerate(lb_rows, 1):
            pname  = str(pname_raw).strip('"')  if pname_raw  is not None else "(unknown)"
            cscore = float(cscore_raw)           if cscore_raw is not None else 0.0
            print(f"    {rank}. {pname:<20}  composite={cscore:.4f}")
        print()

print("\u2713 Per-skill leaderboards generated")

## Step 8 — Save Score Snapshots to DynamoDB

Write one item per Person to the `{PREFIX}-scores` table.

| Attribute | Value |
|---|---|
| `pk` | `PERSON#{id}` |
| `sk` | `SNAPSHOT#{unix_timestamp}` |
| `name` | person display name |
| `raw_score` | expertise_score |
| `normalized_score` | composite_score |
| `topic` | `"composite"` |
| `timestamp` | ISO-8601 UTC |
| `pagerank` | pagerank value |
| `centrality` | centrality value |
| `betweenness` | betweenness value |

In [ ]:
from decimal import Decimal
import datetime

SCORES_TABLE = f"{PREFIX}-scores"
snapshot_ts  = int(time.time())
iso_ts       = datetime.datetime.fromtimestamp(snapshot_ts, tz=datetime.timezone.utc).isoformat()

written = 0
errors  = []

for rec in records:
    person_id = rec["id"]
    if not person_id:
        continue
    try:
        dynamo.put_item(
            TableName=SCORES_TABLE,
            Item={
                "pk":               {"S": f"PERSON#{person_id}"},
                "sk":               {"S": f"SNAPSHOT#{snapshot_ts}"},
                "name":             {"S": rec["name"]},
                "raw_score":        {"N": str(round(rec["expertise"],  6))},
                "normalized_score": {"N": str(round(rec["composite"],  6))},
                "topic":            {"S": "composite"},
                "timestamp":        {"S": iso_ts},
                "pagerank":         {"N": str(round(rec["pagerank"],    10))},
                "centrality":       {"N": str(round(rec["centrality"],  2))},
                "betweenness":      {"N": str(round(rec["betweenness"], 10))},
            },
        )
        written += 1
    except Exception as exc:
        errors.append((person_id, str(exc)))

print(f"\u2713 DynamoDB snapshots written: {written}  (table={SCORES_TABLE})")
print(f"  Timestamp : {iso_ts}  (unix={snapshot_ts})")
if errors:
    print(f"  Errors ({len(errors)}):")
    for pid, msg in errors:
        print(f"    {pid}: {msg}")

## Step 9 — Verification

Re-query all Person nodes from AGE and confirm that all four score properties
(`expertise_score`, `centrality`, `pagerank`, `betweenness`, `composite_score`)
are present.

In [ ]:
sql_verify = f"""
SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$
  MATCH (p:Person)
  RETURN
    p.name               as pname,
    p.expertise_score    as exp,
    p.centrality         as cent,
    p.pagerank           as pr,
    p.betweenness        as betw,
    p.composite_score    as comp
  ORDER BY pname
$$) as t(
  pname ag_catalog.agtype,
  exp   ag_catalog.agtype,
  cent  ag_catalog.agtype,
  pr    ag_catalog.agtype,
  betw  ag_catalog.agtype,
  comp  ag_catalog.agtype
)
"""
verify_rows = _cypher_multi(sql_verify)

TICK = "\u2713"
CROSS = "\u2717"

print(f"{'Name':<22} {'Expertise':<10} {'Centrality':<12} {'PageRank':<10} {'Betweenness':<13} {'Composite':<10}")
print("-" * 78)

all_ok = True
for row in verify_rows:
    pname, exp, cent, pr_val, betw, comp = row
    name = str(pname).strip('"') if pname is not None else "(unknown)"
    flags = [
        TICK if exp   is not None else CROSS,
        TICK if cent  is not None else CROSS,
        TICK if pr_val is not None else CROSS,
        TICK if betw  is not None else CROSS,
        TICK if comp  is not None else CROSS,
    ]
    if CROSS in flags:
        all_ok = False
    print(f"{name:<22} {flags[0]:<10} {flags[1]:<12} {flags[2]:<10} {flags[3]:<13} {flags[4]:<10}")

print()
if all_ok:
    print(f"\u2713 All score properties present on all {len(verify_rows)} Person nodes")
else:
    print(f"\u26a0\ufe0f  Some score properties missing — rerun earlier steps")

## Step 10 — Save Snapshot Timestamp to config.json

Record the scoring run in `config.json` so downstream notebooks (e.g., advisor,
portal data-seeding) can reference the latest snapshot.

In [ ]:
config_path = Path("config.json")
config = json.loads(config_path.read_text())

config["SCORING_SNAPSHOT_TS"]  = snapshot_ts
config["SCORING_SNAPSHOT_ISO"] = iso_ts
config["SCORING_PERSON_COUNT"] = len(records)

config_path.write_text(json.dumps(config, indent=2))

print(f"\u2713 config.json updated")
print(f"  SCORING_SNAPSHOT_TS  = {snapshot_ts}")
print(f"  SCORING_SNAPSHOT_ISO = {iso_ts}")
print(f"  SCORING_PERSON_COUNT = {len(records)}")
print()
print("Next: 15-deploy-runtime.ipynb — Deploy agents to AgentCore Runtime")